# Cost-Sensitive Classification – Heart Disease
### Enterprise 13-Step Standard

In [1]:
# Step 1 – Imports + seed
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, os, joblib
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)
print("Imports OK")


Imports OK


## Step 2 – Data Loading

In [2]:
# Step 2 – Data loading with fallback
csv_path = r'/home/python/03_Custom_addons/ML/supervised_learning/Tree_Based/DecisionTree/DecisionTreeClassifier/data/heart_disease/heart.csv'
df = pd.read_csv(csv_path)


# thal: cast to string so OHE works
df['thal'] = df['thal'].astype(str)


print(df.shape)
df.head(3)


(920, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1


## Step 3 – Preprocessing Pipeline

In [3]:
# Step 3 – Preprocessing
drop_cols = []
target_col = 'target'

df = df.dropna(subset=[target_col])
X = df.drop(columns=[c for c in drop_cols + [target_col] if c in df.columns], errors='ignore')
y = df[target_col]

# Encode target if string
if y.dtype == object:
    le = LabelEncoder()
    y = le.fit_transform(y)
    print('Classes:', le.classes_)

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
print(f'Numeric: {len(num_cols)}  Categorical: {len(cat_cols)}')

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc',  StandardScaler())])

transformers = [('num', num_pipe, num_cols)]
if cat_cols:
    cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                         ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    transformers.append(('cat', cat_pipe, cat_cols))

preprocessor = ColumnTransformer(transformers)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
class_counts = pd.Series(y_train).value_counts()
print('Class distribution (train):')
print(class_counts)
imbalance_ratio = class_counts.max() / class_counts.min()
print(f'Imbalance ratio: {imbalance_ratio:.2f}')


Numeric: 12  Categorical: 1
Train: (736, 13)  Test: (184, 13)
Class distribution (train):
target
1    407
0    329
Name: count, dtype: int64
Imbalance ratio: 1.24


## Step 4 – Algorithm Pipeline Construction

In [4]:
# Step 4 – Build pipelines for each class_weight setting
pipelines = {}
for cw in [None, 'balanced']:
    pipelines[str(cw)] = Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(class_weight=cw, max_iter=1000, random_state=42))
    ])
print('Pipelines constructed:', list(pipelines.keys()))


Pipelines constructed: ['None', 'balanced']


## Step 5 – Baseline Dummy Benchmark

In [5]:
# Step 5 – Dummy classifier baseline
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)
dummy_f1 = f1_score(y_test, y_dummy, average='weighted', zero_division=0)
print(f'Dummy F1 (weighted): {dummy_f1:.4f}')


Dummy F1 (weighted): 0.3954


## Step 6 – Training + Convergence Monitoring

In [6]:
# Step 6 – Train and compare class_weight settings
results = {}
for cw_key, pipe in pipelines.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    elapsed = time.time() - t0
    y_pred = pipe.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    results[cw_key] = {
        'f1': f1,
        'report': classification_report(y_test, y_pred, zero_division=0),
        'y_pred': y_pred,
        'train_time': elapsed
    }
    print(f'class_weight={cw_key}  F1={f1:.4f}  time={elapsed:.2f}s')
    print(results[cw_key]['report'])


class_weight=None  F1=0.8362  time=0.27s
              precision    recall  f1-score   support

           0       0.84      0.78      0.81        82
           1       0.83      0.88      0.86       102

    accuracy                           0.84       184
   macro avg       0.84      0.83      0.83       184
weighted avg       0.84      0.84      0.84       184



class_weight=balanced  F1=0.8261  time=0.19s
              precision    recall  f1-score   support

           0       0.80      0.80      0.80        82
           1       0.84      0.84      0.84       102

    accuracy                           0.83       184
   macro avg       0.82      0.82      0.82       184
weighted avg       0.83      0.83      0.83       184



## Step 7 – Test Set Evaluation

In [7]:
# Step 7 – Full metrics comparison
print('=== class_weight=None ===')
print(results['None']['report'])
print('=== class_weight=balanced ===')
print(results['balanced']['report'])
improvement = results['balanced']['f1'] - results['None']['f1']
print(f'F1 improvement from balancing: {improvement:+.4f}')


=== class_weight=None ===
              precision    recall  f1-score   support

           0       0.84      0.78      0.81        82
           1       0.83      0.88      0.86       102

    accuracy                           0.84       184
   macro avg       0.84      0.83      0.83       184
weighted avg       0.84      0.84      0.84       184

=== class_weight=balanced ===
              precision    recall  f1-score   support

           0       0.80      0.80      0.80        82
           1       0.84      0.84      0.84       102

    accuracy                           0.83       184
   macro avg       0.82      0.82      0.82       184
weighted avg       0.83      0.83      0.83       184

F1 improvement from balancing: -0.0101


## Step 8 – Visualization

In [8]:
# Step 8 – Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (cw_key, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'class_weight={cw_key}\nF1={res["f1"]:.4f}')
plt.tight_layout()
plt.savefig('confusion_matrix_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')


Confusion matrix saved.


## Step 9 – Cross-Validation Stability

In [9]:
# Step 9 – 3-fold CV for balanced pipeline
from sklearn.model_selection import cross_val_score
bal_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(bal_pipe, X_train, y_train, cv=cv,
                            scoring='f1_weighted', n_jobs=-1)
print(f'3-Fold CV F1 (balanced): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('Fold scores:', np.round(cv_scores, 4))


3-Fold CV F1 (balanced): 0.7963 ± 0.0116
Fold scores: [0.7879 0.8128 0.7883]


## Step 10 – Production Serialization

In [10]:
# Step 10 – Production pipeline serialization
prod_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
prod_pipe.fit(X_train, y_train)
joblib_path = 'cs_heart_pipeline.joblib'
joblib.dump(prod_pipe, joblib_path, compress=3)
print(f'Saved: {joblib_path} ({os.path.getsize(joblib_path)/1024:.1f} KB)')
loaded = joblib.load(joblib_path)
print('Reload check OK:', loaded.predict(X_test[:3]))


Saved: cs_heart_pipeline.joblib (2.2 KB)
Reload check OK: [0 1 1]


## Step 11 – Latency Smoke Test

In [11]:
# Step 11 – Latency smoke test (<50ms)
sample = X_test.iloc[:1]
latencies = []
for _ in range(100):
    t0 = time.perf_counter()
    loaded.predict(sample)
    latencies.append((time.perf_counter() - t0) * 1000)
avg_lat = np.mean(latencies)
print(f'Avg latency: {avg_lat:.2f} ms  (max={np.max(latencies):.2f} ms)')
assert avg_lat < 50.0, f'Latency too high: {avg_lat:.2f} ms'
print('✓ Latency OK')


Avg latency: 9.75 ms  (max=27.37 ms)
✓ Latency OK


## Step 12 – Advanced Hyperparameter Tuning / Custom Class Weights

In [12]:
# Step 12 – Manual class weight computation + multi-model comparison
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_dict = dict(zip(classes, weights))
print('Computed class weights:', class_dict)

adv_results = {}
for name, clf in [
    ('LR_balanced',  LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ('RF_balanced',  RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42)),
    ('SVC_balanced', SVC(class_weight='balanced', probability=True, random_state=42)),
    ('LR_custom',    LogisticRegression(class_weight=class_dict, max_iter=1000, random_state=42)),
    ('LR_none',      LogisticRegression(class_weight=None, max_iter=1000, random_state=42)),
]:
    pipe = Pipeline([('pre', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    adv_results[name] = {'f1': f1, 'acc': acc}
    print(f'{name:20s}  F1={f1:.4f}  Acc={acc:.4f}')

adv_df = pd.DataFrame(adv_results).T
print('\nAdvanced comparison:')
print(adv_df.to_string())


Computed class weights: {np.int64(0): np.float64(1.1185410334346504), np.int64(1): np.float64(0.9041769041769042)}


LR_balanced           F1=0.8261  Acc=0.8261


RF_balanced           F1=0.8532  Acc=0.8533
SVC_balanced          F1=0.8524  Acc=0.8533


LR_custom             F1=0.8261  Acc=0.8261
LR_none               F1=0.8362  Acc=0.8370

Advanced comparison:
                    f1       acc
LR_balanced   0.826087  0.826087
RF_balanced   0.853169  0.853261
SVC_balanced  0.852424  0.853261
LR_custom     0.826087  0.826087
LR_none       0.836190  0.836957


## Step 13 – Executive Decision Matrix

| Model | Class Weight | Weighted F1 | Notes |
|-------|-------------|-------------|-------|
| LogisticRegression | None | baseline | No imbalance correction |
| LogisticRegression | balanced | **improved** | Auto-reweights minority |
| RandomForestClassifier | balanced | ensemble boost | Best for nonlinear |
| SVC | balanced | kernel-based | Robust, slower |
| LogisticRegression | custom | manual weights | Fine-grained control |

**Recommendation**: Use `class_weight='balanced'` LogisticRegression or RandomForestClassifier for production on imbalanced Heart Disease data.
